# Designed Pass Classification from Route Trees

Classifies every pass play's **OC design intent** by measuring receiver routes from tracking data, rather than relying on outcome data (which is missing for sacks/scrambles).

**Output**: `data/designed_pass.csv` — one row per pass play with design labels and route features.

**Categories** (depth × direction = 12 labels):
- Depth: `screen` (<3yd), `short` (3-10), `medium` (10-20), `deep` (20+) — based on max route depth
- Direction: `left`, `middle`, `right` — based on weighted lateral movement of 7+ yard routes

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

plays = pd.read_csv('data/plays.csv')
players = pd.read_csv('data/players.csv')

RECEIVER_POSITIONS = {'WR', 'TE', 'RB', 'FB'}
player_pos = players[['nflId', 'position']].copy()

pass_plays = plays[plays['isDropback'] == True][['gameId', 'playId', 'possessionTeam', 'passResult', 'offenseFormation']].copy()
print(f"Total pass plays: {len(pass_plays)}")
print(f"Tracking weeks: 1-9")

In [ ]:
def classify_play_routes(play_tracking, snap_frame, route_end_frame, play_dir, player_pos_df):
    """Measure receiver routes and classify the designed pass.
    
    Returns dict with route features and designedPass label, or None if no receivers found.
    """
    snap_pos = play_tracking[play_tracking['frameId'] == snap_frame].merge(player_pos_df, on='nflId')
    end_pos = play_tracking[play_tracking['frameId'] == route_end_frame].merge(player_pos_df, on='nflId')
    
    recs_snap = snap_pos[snap_pos['position'].isin(RECEIVER_POSITIONS)]
    recs_end = end_pos[end_pos['position'].isin(RECEIVER_POSITIONS)]
    
    if len(recs_snap) == 0 or len(recs_end) == 0:
        return None
    
    merged = recs_snap[['nflId', 'position', 'x', 'y']].merge(
        recs_end[['nflId', 'x', 'y']], on='nflId', suffixes=('_snap', '_end'))
    
    if len(merged) == 0:
        return None
    
    # Compute depth (downfield) and lateral displacement
    if play_dir == 'right':
        merged['depth'] = merged['x_end'] - merged['x_snap']
        merged['lateral'] = merged['y_end'] - merged['y_snap']
    else:
        merged['depth'] = merged['x_snap'] - merged['x_end']
        merged['lateral'] = merged['y_snap'] - merged['y_end']
    
    max_depth = merged['depth'].max()
    avg_depth = merged['depth'].mean()
    
    # Count routes by depth tier
    n_deep = int((merged['depth'] >= 15).sum())
    n_mid = int(((merged['depth'] >= 7) & (merged['depth'] < 15)).sum())
    n_short = int(((merged['depth'] >= 0) & (merged['depth'] < 7)).sum())
    n_behind = int((merged['depth'] < 0).sum())
    
    # Direction: weighted lateral movement of routes 7+ yards deep
    deep_routes = merged[merged['depth'] >= 7]
    if len(deep_routes) > 0:
        weighted_lat = (deep_routes['lateral'] * deep_routes['depth']).sum() / deep_routes['depth'].sum()
        if weighted_lat < -3:
            direction = 'left'
        elif weighted_lat > 3:
            direction = 'right'
        else:
            direction = 'middle'
    else:
        avg_lat = merged['lateral'].mean()
        if avg_lat < -2:
            direction = 'left'
        elif avg_lat > 2:
            direction = 'right'
        else:
            direction = 'middle'
    
    # Depth category from max route depth
    if max_depth < 3:
        depth_cat = 'screen'
    elif max_depth < 10:
        depth_cat = 'short'
    elif max_depth < 20:
        depth_cat = 'medium'
    else:
        depth_cat = 'deep'
    
    return {
        'max_route_depth': round(max_depth, 1),
        'avg_route_depth': round(avg_depth, 1),
        'route_direction': direction,
        'depth_category': depth_cat,
        'designedPass': f'{depth_cat}_{direction}',
        'n_deep_routes': n_deep,
        'n_mid_routes': n_mid,
        'n_short_routes': n_short,
        'n_behind_routes': n_behind,
        'n_receivers': len(merged),
        'frames_to_throw': route_end_frame - snap_frame,
    }

print("classify_play_routes defined")

In [ ]:
def process_week(week_num, pass_plays_df, player_pos_df):
    """Process one week of tracking data. Returns list of result dicts."""
    path = f'data/tracking_week_{week_num}.csv'
    print(f"Loading {path}...")
    tracking = pd.read_csv(path)
    
    # Only keep offensive players on pass plays to reduce memory
    available = tracking[['gameId', 'playId']].drop_duplicates()
    week_pass = pass_plays_df.merge(available, on=['gameId', 'playId'])
    print(f"  Week {week_num}: {len(week_pass)} pass plays with tracking")
    
    results = []
    skipped = 0
    
    for _, play in week_pass.iterrows():
        gid, pid = play['gameId'], play['playId']
        off_team = play['possessionTeam']
        
        pt = tracking[(tracking['gameId'] == gid) & (tracking['playId'] == pid)]
        off_pt = pt[pt['club'] == off_team]
        
        if len(off_pt) == 0:
            skipped += 1
            continue
        
        # Find snap frame
        snap = pt[pt['event'] == 'ball_snap']
        if len(snap) == 0:
            skipped += 1
            continue
        snap_frame = snap['frameId'].iloc[0]
        play_dir = off_pt['playDirection'].iloc[0]
        
        # Find route end: pass_forward > qb_sack > snap+30
        pass_fwd = pt[pt['event'] == 'pass_forward']
        sack_evt = pt[pt['event'] == 'qb_sack']
        
        if len(pass_fwd) > 0:
            route_end = pass_fwd['frameId'].iloc[0]
        elif len(sack_evt) > 0:
            route_end = sack_evt['frameId'].iloc[0]
        else:
            route_end = min(snap_frame + 30, pt['frameId'].max())
        
        result = classify_play_routes(off_pt, snap_frame, route_end, play_dir, player_pos_df)
        if result is None:
            skipped += 1
            continue
        
        result['gameId'] = gid
        result['playId'] = pid
        results.append(result)
    
    print(f"  Classified {len(results)} plays, skipped {skipped}")
    return results

print("process_week defined")

## Process all 9 weeks
~7.7GB of tracking data, processed one week at a time to manage memory.

In [ ]:
all_results = []

for week in range(1, 10):
    week_results = process_week(week, pass_plays, player_pos)
    all_results.extend(week_results)
    print()

designed = pd.DataFrame(all_results)
print(f"Total classified: {len(designed)} / {len(pass_plays)} pass plays")
print(f"\n=== designedPass categories ===")
print(designed['designedPass'].value_counts().to_string())

## Merge with plays.csv and save

In [ ]:
# Merge route features onto plays.csv (left join — keeps all plays, NaN for non-pass/missing)
enriched = plays.merge(designed, on=['gameId', 'playId'], how='left')

print(f"Enriched plays: {len(enriched)}")
print(f"Plays with designedPass: {enriched['designedPass'].notna().sum()}")
print(f"Plays without (runs + missing): {enriched['designedPass'].isna().sum()}")

# Sanity check: designed depth vs actual passLength for thrown passes
thrown = enriched[enriched['passResult'].isin(['C', 'I', 'IN']) & enriched['depth_category'].notna()]
print(f"\n=== Validation: actual passLength by designed depth ===")
for cat in ['screen', 'short', 'medium', 'deep']:
    sub = thrown[thrown['depth_category'] == cat]
    if len(sub) > 0:
        print(f"  {cat}: n={len(sub)}, avg passLength={sub['passLength'].mean():.1f}, median={sub['passLength'].median():.1f}")

In [ ]:
# Cross-tab: designedPass vs passResult — shows how sacks skew toward deep designs
print("=== designedPass depth × passResult ===")
print(pd.crosstab(enriched['depth_category'], enriched['passResult'], margins=True))

print("\n=== designedPass direction × passResult ===")
print(pd.crosstab(enriched['route_direction'], enriched['passResult'], margins=True))

In [ ]:
# Save
enriched.to_csv('data/designed_pass.csv', index=False)
print(f"Saved data/designed_pass.csv ({len(enriched)} rows, {len(enriched.columns)} columns)")
print(f"\nNew columns added: {list(designed.columns.drop(['gameId', 'playId']))}")